# Spa Bookings: Labor Forecasting & Intelligent Scheduling

**Use Cases:**
1. **Labor Forecasting** - Predict appointment demand by location & service category using `SNOWFLAKE.ML.FORECAST`
2. **Intelligent Scheduling** - Replace rule-based scheduling with a Cortex Agent that reasons across forecasts, provider skills, and availability

**Key Snowflake Features:** ML.FORECAST (time-series), Cortex Agent, Semantic View, Custom Tool (Stored Procedure)

## 1. Source Data Overview

In [ ]:
SELECT
    COUNT(*) AS total_appointments,
    COUNT(DISTINCT CLIENT_ID) AS unique_clients,
    COUNT(DISTINCT PROVIDER_ID) AS providers,
    COUNT(DISTINCT LOCATION_CITY) AS locations,
    COUNT(DISTINCT SERVICE_CATEGORY) AS categories,
    MIN(APPOINTMENT_DATE) AS data_start,
    MAX(APPOINTMENT_DATE) AS data_end
FROM SPABOOKINGS.PUBLIC.APPOINTMENTS

In [ ]:
SELECT LOCATION_CITY, SERVICE_CATEGORY, SUM(APPOINTMENT_COUNT) AS total_appts
FROM SPABOOKINGS.PUBLIC.DAILY_APPOINTMENT_VOLUME
GROUP BY 1, 2
ORDER BY total_appts DESC
LIMIT 15

## 2. Provider Skills & Availability

In [ ]:
SELECT PROVIDER_ID, PROVIDER_NAME, LOCATION_CITY, SKILL_CATEGORIES, MAX_DAILY_HOURS, HOURLY_RATE
FROM SPABOOKINGS.PUBLIC.PROVIDER_SKILLS
ORDER BY PROVIDER_ID

In [ ]:
SELECT ps.PROVIDER_NAME, pa.DAY_NAME, pa.AVAILABLE_START, pa.AVAILABLE_END, pa.IS_AVAILABLE
FROM SPABOOKINGS.PUBLIC.PROVIDER_AVAILABILITY pa
JOIN SPABOOKINGS.PUBLIC.PROVIDER_SKILLS ps ON pa.PROVIDER_ID = ps.PROVIDER_ID
WHERE ps.PROVIDER_ID IN ('PRV0001', 'PRV0002', 'PRV0003')
ORDER BY ps.PROVIDER_NAME, pa.DAY_OF_WEEK

## 3. Labor Demand Forecasting

Train a multi-series forecast model using `SNOWFLAKE.ML.FORECAST` -- **one SQL statement** predicts demand for all 50 location/category combinations.

In [ ]:
CREATE OR REPLACE VIEW SPABOOKINGS.PUBLIC.V_DAILY_DEMAND AS
SELECT
    [LOCATION_CITY, SERVICE_CATEGORY] AS SERIES_KEY,
    APPOINTMENT_DATE::TIMESTAMP_NTZ AS TS,
    COUNT(*)::FLOAT AS APPOINTMENT_COUNT
FROM SPABOOKINGS.PUBLIC.APPOINTMENTS
WHERE APPOINTMENT_STATUS = 'completed'
GROUP BY 1, 2

In [ ]:
-- One SQL statement trains 50 forecast models (one per location/category)!
CREATE OR REPLACE SNOWFLAKE.ML.FORECAST SPABOOKINGS.PUBLIC.LABOR_DEMAND_FORECAST(
    INPUT_DATA => TABLE(SPABOOKINGS.PUBLIC.V_DAILY_DEMAND),
    SERIES_COLNAME => 'SERIES_KEY',
    TIMESTAMP_COLNAME => 'TS',
    TARGET_COLNAME => 'APPOINTMENT_COUNT'
)

In [ ]:
-- Generate 14-period forecast for all series
CREATE OR REPLACE TABLE SPABOOKINGS.PUBLIC.LABOR_FORECAST_RESULTS AS
SELECT * FROM TABLE(SPABOOKINGS.PUBLIC.LABOR_DEMAND_FORECAST!FORECAST(FORECASTING_PERIODS => 14))

## 4. Forecast Results & Evaluation

In [ ]:
SELECT SERIES, TS::DATE AS FORECAST_DATE,
       ROUND(FORECAST, 1) AS PREDICTED_DEMAND,
       ROUND(LOWER_BOUND, 1) AS LOWER,
       ROUND(UPPER_BOUND, 1) AS UPPER
FROM SPABOOKINGS.PUBLIC.LABOR_FORECAST_RESULTS
WHERE SERIES[0]::VARCHAR = 'San Francisco'
ORDER BY SERIES, TS

In [ ]:
-- Model evaluation metrics (auto cross-validated)
SELECT SERIES, ERROR_METRIC, ROUND(METRIC_VALUE, 4) AS VALUE
FROM TABLE(SPABOOKINGS.PUBLIC.LABOR_DEMAND_FORECAST!SHOW_EVALUATION_METRICS())
WHERE ERROR_METRIC = 'MAE'
ORDER BY METRIC_VALUE DESC
LIMIT 10

## 5. Visualize Demand Forecast

In [ ]:
import pandas as pd
import matplotlib.pyplot as plt
from snowflake.snowpark.context import get_active_session

session = get_active_session()

hist = session.sql("""
    SELECT APPOINTMENT_DATE AS DT, COUNT(*) AS CNT
    FROM SPABOOKINGS.PUBLIC.APPOINTMENTS
    WHERE APPOINTMENT_STATUS = 'completed' AND LOCATION_CITY = 'San Francisco' AND SERVICE_CATEGORY = 'Hair'
    GROUP BY 1 ORDER BY 1
""").to_pandas()

fcast = session.sql("""
    SELECT TS::DATE AS DT, FORECAST AS CNT, LOWER_BOUND, UPPER_BOUND
    FROM SPABOOKINGS.PUBLIC.LABOR_FORECAST_RESULTS
    WHERE SERIES[0]::VARCHAR = 'San Francisco' AND SERIES[1]::VARCHAR = 'Hair'
    ORDER BY TS
""").to_pandas()

fig, ax = plt.subplots(figsize=(14, 5))
ax.plot(pd.to_datetime(hist['DT']), hist['CNT'], color='#29B5E8', alpha=0.6, label='Historical')
ax.plot(pd.to_datetime(fcast['DT']), fcast['CNT'], color='#e74c3c', linewidth=2, label='Forecast')
ax.fill_between(pd.to_datetime(fcast['DT']), fcast['LOWER_BOUND'], fcast['UPPER_BOUND'], alpha=0.2, color='#e74c3c')
ax.set_title('San Francisco - Hair: Demand Forecast', fontsize=14, fontweight='bold')
ax.set_xlabel('Date')
ax.set_ylabel('Appointments')
ax.legend()
plt.tight_layout()
plt.show()

## 6. Staffing Recommendations

In [ ]:
-- Call the stored procedure to identify staffing gaps
CALL SPABOOKINGS.PUBLIC.RECOMMEND_SCHEDULE('2026-04-09', '2026-04-16')

In [ ]:
-- Focus on understaffed locations
SELECT * FROM TABLE(RESULT_SCAN(LAST_QUERY_ID()))
WHERE STAFFING_STATUS = 'Understaffed'
ORDER BY FORECAST_DATE, LOCATION_CITY

## 7. Cortex Agent - Intelligent Scheduling

The agent combines Cortex Analyst (semantic view) with the scheduling procedure as a custom tool.
Salon managers can ask natural language questions like:
- *"What's the forecast for next week in Austin?"*
- *"Are we understaffed anywhere?"*
- *"Which providers can do Hair and Skin?"*

In [ ]:
SHOW AGENTS IN SCHEMA SPABOOKINGS.PUBLIC

In [ ]:
DESCRIBE AGENT SPABOOKINGS.PUBLIC.SCHEDULING_AGENT

## Summary

| Component | Snowflake Feature | Purpose |
|-----------|-------------------|---------|
| **Demand Forecast** | `SNOWFLAKE.ML.FORECAST` | Predict appointment volume by location + category |
| **Provider Data** | Tables | Skills, availability, capacity per provider |
| **Staffing Logic** | Stored Procedure | Compare forecast vs available staff, flag gaps |
| **Semantic View** | `CREATE SEMANTIC VIEW` | Business-friendly data layer for Cortex Analyst |
| **Scheduling Agent** | `CREATE AGENT` | Natural language interface for salon managers |

**Key Differentiators vs. a rule-based scheduler:**
- Demand-aware (knows busy/slow periods via ML forecast)
- Multi-skill aware (matches provider skills to service demand)
- Conversational (managers ask questions, get answers)
- All within Snowflake (HIPAA compliant, no data movement)